# 🔴 SpendLens — Parloa Strategic Procurement Analysis

**Pure Python dashboard** using `plotly` and `pandas`.

This notebook predicts Parloa's procurement spend across 10 categories, tracks growth trends,
and flags bottleneck risks (supplier concentration, single-source dependencies, contract clustering).

### Parloa at a glance
| Metric | Value |
|--------|-------|
| Employees | ~430 (Feb 2026), target 600 by EOY 2026 |
| ARR | $50M+ (Dec 2025) |
| Valuation | $3B (Series D, Jan 2026) |
| Total Funding | $560M+ |
| Offices | Berlin (HQ), Munich, NYC, expanding SF + Madrid |
| Product | AI Agent Management Platform for enterprise contact centers |

## 0. Setup
Run this cell once to install the required packages.

In [1]:
# Run this once — installs plotly and pandas
!pip install plotly pandas --quiet

## 1. Imports & Configuration

In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# If charts don't render, uncomment the line below:
# import plotly.io as pio; pio.renderers.default = 'notebook'

print('✅ Imports loaded')

✅ Imports loaded


## 2. Data Model — Parloa Spend Predictions

Each category has:
- **spend**: estimated annual spend in €K for 2022–2026
- **concentration**: % of spend going to top supplier (higher = riskier)
- **risk**: overall risk level (Critical / High / Medium / Low)
- **single_source**: whether there's a dangerous single-supplier dependency
- **bottleneck**: what makes this category risky for procurement

💡 *Try changing the spend numbers or adding new categories — all charts update automatically!*

In [3]:
YEARS = [2022, 2023, 2024, 2025, 2026]
HEADCOUNT = [49, 120, 220, 380, 600]

categories_raw = [
    {
        "name": "Cloud & Compute Infrastructure",
        "desc": "AWS/GCP/Azure, GPU instances, CDN, storage",
        "spend": [4200, 7800, 12500, 17800, 24000],
        "suppliers": ["AWS", "Google Cloud", "Azure"],
        "concentration": 72,
        "risk": "Critical",
        "single_source": True,
        "bottleneck": "72% on primary cloud provider. GPU capacity constraints during peak training.",
        "lead_time_days": 0,
        "contract_end": "2026-09",
    },
    {
        "name": "AI/ML APIs & Data Services",
        "desc": "LLM providers (OpenAI, Anthropic), training data, annotation",
        "spend": [800, 2200, 4800, 6500, 9200],
        "suppliers": ["OpenAI", "Anthropic", "Scale AI", "Labelbox"],
        "concentration": 55,
        "risk": "High",
        "single_source": False,
        "bottleneck": "LLM pricing volatility. Token cost directly impacts COGS.",
        "lead_time_days": 14,
        "contract_end": "2026-06",
    },
    {
        "name": "IT Software & SaaS",
        "desc": "Dev tools, security, monitoring (GitHub, Datadog, Okta, Slack)",
        "spend": [900, 1400, 2200, 3100, 4200],
        "suppliers": ["GitHub", "Datadog", "Atlassian", "Okta", "Slack"],
        "concentration": 22,
        "risk": "Low",
        "single_source": False,
        "bottleneck": "License scaling costs with 40% headcount growth. Shadow IT in new offices.",
        "lead_time_days": 7,
        "contract_end": "Various",
    },
    {
        "name": "Recruitment & Talent Acquisition",
        "desc": "Agencies, job boards, employer branding, relocation",
        "spend": [600, 1100, 2400, 4200, 6800],
        "suppliers": ["Personio", "LinkedIn", "Greenhouse", "External Agencies"],
        "concentration": 35,
        "risk": "High",
        "single_source": False,
        "bottleneck": "Hiring 170+ roles (380 to 600). Agency fees 20-25% for senior AI talent.",
        "lead_time_days": 45,
        "contract_end": "2026-12",
    },
    {
        "name": "Professional Services",
        "desc": "Legal, audit, consulting, EU AI Act compliance, IP",
        "spend": [400, 700, 1200, 2100, 3200],
        "suppliers": ["Big 4 Audit", "Baker McKenzie", "Compliance Advisors"],
        "concentration": 40,
        "risk": "Medium",
        "single_source": False,
        "bottleneck": "EU AI Act compliance accelerating. Multi-jurisdiction legal (DE, US, ES).",
        "lead_time_days": 21,
        "contract_end": "2026-03",
    },
    {
        "name": "Marketing & Events",
        "desc": "Conferences, digital campaigns, PR, partner events",
        "spend": [300, 800, 1800, 3500, 5500],
        "suppliers": ["Event Agencies", "Google Ads", "PR Firms", "HubSpot"],
        "concentration": 28,
        "risk": "Medium",
        "single_source": False,
        "bottleneck": "New CMO restructuring spend. US market entry needs 3x budget vs DACH.",
        "lead_time_days": 30,
        "contract_end": "Various",
    },
    {
        "name": "Facilities & Office",
        "desc": "Berlin HQ, Munich, NYC midtown, new SF + Madrid",
        "spend": [500, 900, 1500, 2800, 4800],
        "suppliers": ["Landlords", "WeWork/Flex", "Office Suppliers"],
        "concentration": 45,
        "risk": "High",
        "single_source": False,
        "bottleneck": "2 new office buildouts (SF, Madrid) in 2026. Lease overlap costs.",
        "lead_time_days": 90,
        "contract_end": "2028-06",
    },
    {
        "name": "Travel & Entertainment",
        "desc": "Multi-office travel, customer visits, team offsites",
        "spend": [200, 500, 1100, 2000, 3200],
        "suppliers": ["TMC", "Airlines", "Hotels", "Navan"],
        "concentration": 30,
        "risk": "Low",
        "single_source": False,
        "bottleneck": "5-office structure driving 80% YoY travel growth.",
        "lead_time_days": 3,
        "contract_end": "2026-09",
    },
    {
        "name": "Hardware & Equipment",
        "desc": "Laptops, monitors, GPU servers, networking gear",
        "spend": [300, 500, 900, 1500, 2400],
        "suppliers": ["Apple", "Dell", "NVIDIA", "CDW"],
        "concentration": 38,
        "risk": "Medium",
        "single_source": False,
        "bottleneck": "GPU lead times 8-16 weeks. Onboarding kits for 170+ new hires.",
        "lead_time_days": 56,
        "contract_end": "N/A",
    },
    {
        "name": "Telco & Voice Infrastructure",
        "desc": "SIP trunking, phone numbers, voice APIs (product-critical)",
        "spend": [400, 800, 1400, 2200, 3000],
        "suppliers": ["Twilio", "Vonage", "Deutsche Telekom", "Bandwidth"],
        "concentration": 65,
        "risk": "Critical",
        "single_source": True,
        "bottleneck": "Voice infra is product-critical. 65% on single SIP provider.",
        "lead_time_days": 30,
        "contract_end": "2026-07",
    },
]

print(f'✅ {len(categories_raw)} categories defined across {len(YEARS)} years')

✅ 10 categories defined across 5 years


## 3. Build DataFrames

We convert the raw data into **pandas DataFrames** — this is the foundation for all analysis.

Key Python concepts here:
- `list comprehension` to build rows
- `pd.DataFrame()` to create tables
- `.groupby()` and `.merge()` to aggregate and join data

In [4]:
# --- Spend data: one row per category per year ---
rows = []
for cat in categories_raw:
    for i, year in enumerate(YEARS):
        rows.append({
            "category": cat["name"],
            "year": year,
            "spend_k": cat["spend"][i],
        })

df_spend = pd.DataFrame(rows)

# --- Category metadata ---
df_meta = pd.DataFrame([{
    "category": c["name"],
    "concentration": c["concentration"],
    "risk": c["risk"],
    "single_source": c["single_source"],
    "bottleneck": c["bottleneck"],
    "lead_time_days": c["lead_time_days"],
    "contract_end": c["contract_end"],
    "suppliers": ", ".join(c["suppliers"]),
    "spend_2026e": c["spend"][4],
} for c in categories_raw])

# --- Headcount ---
df_headcount = pd.DataFrame({'year': YEARS, 'headcount': HEADCOUNT})

# --- Totals per year ---
df_totals = df_spend.groupby('year')['spend_k'].sum().reset_index()
df_totals.columns = ['year', 'total_spend_k']
df_totals = df_totals.merge(df_headcount, on='year')
df_totals['spend_per_head_k'] = (df_totals['total_spend_k'] / df_totals['headcount']).round(1)

# --- CAGR per category (Compound Annual Growth Rate) ---
# Formula: CAGR = (end_value / start_value) ^ (1/years) - 1
for c in categories_raw:
    cagr = ((c['spend'][4] / c['spend'][0]) ** (1/4) - 1) * 100
    df_meta.loc[df_meta['category'] == c['name'], 'cagr'] = round(cagr, 1)

print('✅ DataFrames ready!')
print(f'   df_spend:  {df_spend.shape[0]} rows × {df_spend.shape[1]} cols')
print(f'   df_meta:   {df_meta.shape[0]} rows × {df_meta.shape[1]} cols')
print(f'   df_totals: {df_totals.shape[0]} rows × {df_totals.shape[1]} cols')

✅ DataFrames ready!
   df_spend:  50 rows × 3 cols
   df_meta:   10 rows × 10 cols
   df_totals: 5 rows × 4 cols


### 3a. Explore the data

💡 *In Jupyter, the last expression in a cell gets displayed automatically — no `print()` needed!*

In [5]:
# Look at the spend data
df_spend.head(10)

,category,year,spend_k
0,Cloud & Compute Infrastructure,2022,4200
1,Cloud & Compute Infrastructure,2023,7800
2,Cloud & Compute Infrastructure,2024,12500
3,Cloud & Compute Infrastructure,2025,17800
4,Cloud & Compute Infrastructure,2026,24000
5,AI/ML APIs & Data Services,2022,800
6,AI/ML APIs & Data Services,2023,2200
7,AI/ML APIs & Data Services,2024,4800
8,AI/ML APIs & Data Services,2025,6500
9,AI/ML APIs & Data Services,2026,9200


In [6]:
# Category summary sorted by 2026E spend
df_meta[['category', 'spend_2026e', 'cagr', 'concentration', 'risk', 'single_source']]\
    .sort_values('spend_2026e', ascending=False)

,category,spend_2026e,cagr,concentration,risk,single_source
0,Cloud & Compute Infrastructure,24000,54.6,72,Critical,True
1,AI/ML APIs & Data Services,9200,84.2,55,High,False
3,Recruitment & Talent Acquisition,6800,83.5,35,High,False
5,Marketing & Events,5500,106.9,28,Medium,False
6,Facilities & Office,4800,76.0,45,High,False
2,IT Software & SaaS,4200,47.0,22,Low,False
4,Professional Services,3200,68.2,40,Medium,False
7,Travel & Entertainment,3200,100.0,30,Low,False
9,Telco & Voice Infrastructure,3000,65.5,65,Critical,True
8,Hardware & Equipment,2400,68.2,38,Medium,False


In [7]:
# Total spend per year
df_totals

,year,total_spend_k,headcount,spend_per_head_k
0,2022,8600,49,175.5
1,2023,16700,120,139.2
2,2024,29800,220,135.5
3,2025,45700,380,120.3
4,2026,66300,600,110.5


## 4. Chart Theme

We define colors and a reusable layout so all charts look consistent.

In [8]:
# Color palette
COLORS = ['#FF6B4A', '#3B82F6', '#34D399', '#FBBF24', '#A78BFA',
          '#F472B6', '#FB923C', '#06B6D4', '#E879F9', '#84CC16']

BG = '#0B0E14'
CARD = '#141820'
GRID = '#1E2430'
TEXT = '#E4E8F1'
DIM = '#6B7A94'

RISK_COLORS = {'Critical': '#EF4444', 'High': '#FBBF24', 'Medium': '#FB923C', 'Low': '#34D399'}

# Map each category to a color
cat_colors = {cat['name']: COLORS[i] for i, cat in enumerate(categories_raw)}

# Reusable layout settings (saves repeating ourselves)
LAYOUT = dict(
    paper_bgcolor=BG,
    plot_bgcolor=CARD,
    font=dict(family='Arial, sans-serif', color=TEXT, size=12),
    margin=dict(l=60, r=30, t=50, b=40),
    xaxis=dict(gridcolor=GRID, linecolor=GRID),
    yaxis=dict(gridcolor=GRID, linecolor=GRID),
    legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(size=10)),
)

print('🎨 Theme configured')

🎨 Theme configured


---
## 5. Spend Forecast Charts

Now the fun part — let's visualize everything.

### 📊 Chart 1: Total Spend vs Headcount
Dual-axis chart — bars for spend, dashed line for headcount.
Notice how spend grows faster than headcount (spend per head increases).

In [9]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

# Spend bars
fig.add_trace(
    go.Bar(
        x=df_totals['year'],
        y=df_totals['total_spend_k'],
        name='Total Spend (€K)',
        marker_color='#FF6B4A',
        opacity=0.85,
        text=[f'€{v:,.0f}K' for v in df_totals['total_spend_k']],
        textposition='outside',
    ),
    secondary_y=False,
)

# Headcount line
fig.add_trace(
    go.Scatter(
        x=df_totals['year'],
        y=df_totals['headcount'],
        name='Headcount',
        mode='lines+markers+text',
        line=dict(color='#3B82F6', width=3, dash='dot'),
        marker=dict(size=8),
        text=[str(h) for h in df_totals['headcount']],
        textposition='top center',
    ),
    secondary_y=True,
)

fig.update_layout(title='Total Procurement Spend vs Headcount', **LAYOUT, height=420)
fig.update_yaxes(title_text='Spend (€K)', secondary_y=False, gridcolor=GRID)
fig.update_yaxes(title_text='Headcount', secondary_y=True, gridcolor=GRID)
fig.show()

### 📊 Chart 2: Category Spend Evolution (Stacked Area)
Shows how each category's spend stacks up over time.
Cloud & Compute dominates and is growing fast.

In [10]:
fig = go.Figure()

sorted_cats = df_meta.sort_values('spend_2026e', ascending=True)['category'].tolist()

for cat_name in sorted_cats:
    cat_data = df_spend[df_spend['category'] == cat_name]
    fig.add_trace(go.Scatter(
        x=cat_data['year'],
        y=cat_data['spend_k'],
        name=cat_name.split('&')[0].strip()[:25],
        mode='lines',
        stackgroup='one',
        line=dict(width=0.5, color=cat_colors[cat_name]),
    ))

fig.update_layout(title='Category Spend Evolution (Stacked)', **LAYOUT, height=450, yaxis_title='Spend (€K)')
fig.show()

### 📊 Chart 3: 2026E Spend Treemap
Size = spend amount, Color = supplier concentration.
Red = dangerous concentration, green = healthy diversification.

In [11]:
df_tree = df_meta.sort_values('spend_2026e', ascending=False).copy()
df_tree['label'] = df_tree.apply(
    lambda r: f"{r['category']}\n€{r['spend_2026e']:,.0f}K | {r['risk']} risk", axis=1
)

fig = px.treemap(
    df_tree,
    path=['label'],
    values='spend_2026e',
    color='concentration',
    color_continuous_scale=['#34D399', '#FBBF24', '#EF4444'],
    range_color=[0, 80],
)

fig.update_layout(
    title='2026E Spend (size) × Supplier Concentration (color)',
    paper_bgcolor=BG, font=dict(color=TEXT), height=450,
    coloraxis_colorbar=dict(title='Concentration %'),
)
fig.show()

### 📊 Chart 4: Category CAGR
Compound Annual Growth Rate from 2022 to 2026E.
Marketing & Travel have the highest growth rates.

In [12]:
df_cagr = df_meta.sort_values('cagr', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_cagr['category'],
    x=df_cagr['cagr'],
    orientation='h',
    marker_color=[cat_colors[c] for c in df_cagr['category']],
    text=[f'{v:.1f}%' for v in df_cagr['cagr']],
    textposition='outside',
))

fig.update_layout(title='Category CAGR (2022 → 2026E)', **LAYOUT, height=420, xaxis_title='CAGR %')
fig.show()

---
## 6. Bottleneck Analysis

This is where it gets strategic. We identify:
- **Supplier concentration risk** — too much spend with one vendor
- **Single-source dependencies** — no alternative if supplier fails
- **Lead time exposure** — how fast can you switch?
- **Contract renewal clustering** — too many big renewals at once

### ⚠️ Bottleneck Bubble Map
- **X-axis**: Supplier concentration (higher = riskier)
- **Y-axis**: Annual spend (higher = more impact)
- **Bubble size**: Lead time in days (bigger = slower to switch)
- **Color**: Risk level

Categories in the top-right "danger zone" need immediate attention.

In [13]:
fig = go.Figure()

for _, row in df_meta.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['concentration']],
        y=[row['spend_2026e']],
        mode='markers+text',
        marker=dict(
            size=row['lead_time_days'] + 15,
            color=RISK_COLORS[row['risk']],
            opacity=0.7,
            line=dict(width=2, color=RISK_COLORS[row['risk']]),
        ),
        text=[row['category'].split('&')[0].strip()[:18]],
        textposition='top center',
        textfont=dict(size=9, color=TEXT),
        name=row['category'],
        hovertemplate=(
            f"<b>{row['category']}</b><br>"
            f"Concentration: {row['concentration']}%<br>"
            f"Spend: €{row['spend_2026e']:,.0f}K<br>"
            f"Lead Time: {row['lead_time_days']}d<br>"
            f"Risk: {row['risk']}<extra></extra>"
        ),
    ))

# Danger zone
fig.add_shape(
    type='rect', x0=50, x1=100, y0=0, y1=30000,
    fillcolor='rgba(239,68,68,0.06)',
    line=dict(color='rgba(239,68,68,0.3)', dash='dot'),
)
fig.add_annotation(
    x=75, y=28000, text='⚠ DANGER ZONE',
    showarrow=False, font=dict(size=11, color='#EF4444'),
)

fig.update_layout(
    title='Bottleneck Map — Concentration vs Spend (bubble = lead time)',
    **LAYOUT, height=500, showlegend=False,
    xaxis_title='Supplier Concentration (%)',
    yaxis_title='2026E Spend (€K)',
)
fig.show()

### 🎯 Risk Scorecard Gauges

In [14]:
avg_conc = df_meta['concentration'].mean()
criticals = len(df_meta[df_meta['risk'] == 'Critical'])
single_src = int(df_meta['single_source'].sum())
avg_lead = df_meta['lead_time_days'].mean()
overall = round(avg_conc * 0.4 + criticals * 15 + single_src * 10 + avg_lead * 0.3)

fig = make_subplots(
    rows=1, cols=5,
    specs=[[{'type': 'indicator'}] * 5],
    subplot_titles=['Overall Risk', 'Avg Concentration', 'Critical Count', 'Single-Source', 'Avg Lead Time'],
)

gauges = [
    (overall, 100, 'Score'),
    (avg_conc, 100, '%'),
    (criticals, 5, 'cats'),
    (single_src, 5, 'cats'),
    (avg_lead, 90, 'days'),
]

for i, (val, ref, unit) in enumerate(gauges):
    color = '#EF4444' if (val/ref) > 0.6 else '#FBBF24' if (val/ref) > 0.35 else '#34D399'
    fig.add_trace(
        go.Indicator(
            mode='gauge+number',
            value=val,
            number=dict(font=dict(size=28, color=color)),
            gauge=dict(
                axis=dict(range=[0, ref]),
                bar=dict(color=color),
                bgcolor=CARD,
                steps=[
                    dict(range=[0, ref*0.35], color='rgba(52,211,153,0.1)'),
                    dict(range=[ref*0.35, ref*0.6], color='rgba(251,191,36,0.1)'),
                    dict(range=[ref*0.6, ref], color='rgba(239,68,68,0.1)'),
                ],
            ),
        ),
        row=1, col=i+1,
    )

fig.update_layout(title='Bottleneck Risk Scorecard', paper_bgcolor=BG, font=dict(color=TEXT), height=280)
fig.show()

### 📅 Contract Renewal Timeline
Flags when major contracts expire — clustering = bandwidth risk for procurement team.

In [15]:
renewals = df_meta[
    ~df_meta['contract_end'].isin(['Various', 'N/A'])
].sort_values('contract_end')

fig = go.Figure()

for _, row in renewals.iterrows():
    short = row['category'].split('&')[0].strip()[:20]
    fig.add_trace(go.Bar(
        y=[short],
        x=[row['spend_2026e']],
        orientation='h',
        marker_color=RISK_COLORS[row['risk']],
        text=[f"Expires: {row['contract_end']}  |  €{row['spend_2026e']:,.0f}K"],
        textposition='inside',
        textfont=dict(size=11, color='white'),
        showlegend=False,
    ))

fig.add_annotation(
    text='⚠ 3 contracts (Cloud + Telco + AI/ML = €36.2M) expire Q2-Q3 2026',
    xref='paper', yref='paper', x=0.5, y=-0.2,
    showarrow=False, font=dict(size=11, color='#FBBF24'),
)

fig.update_layout(
    title='Contract Renewal Timeline — Spend at Risk',
    **LAYOUT, height=350, xaxis_title='Annual Spend (€K)',
)
fig.show()

### 📋 Full Bottleneck Summary Table

In [18]:
# Display as a styled pandas table
risk_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}

summary = df_meta[['category', 'spend_2026e', 'concentration', 'risk',
                    'single_source', 'lead_time_days', 'contract_end', 'bottleneck']].copy()
summary['risk_sort'] = summary['risk'].map(risk_order)
summary = summary.sort_values('risk_sort').drop('risk_sort', axis=1)
summary.columns = ['Category', 'Spend 2026E (€K)', 'Concentration %', 'Risk',
                    'Single Source', 'Lead Time (days)', 'Contract End', 'Bottleneck']

summary.style.format({'Spend 2026E (€K)': '€{:,.0f}K', 'Concentration %': '{}%'})

,Category,Spend 2026E (€K),Concentration %,Risk,Single Source,Lead Time (days),Contract End,Bottleneck
0,Cloud & Compute Infrastructure,"€24,000K",72%,Critical,True,0,2026-09,72% on primary cloud provider. GPU capacity constraints during peak training.
9,Telco & Voice Infrastructure,"€3,000K",65%,Critical,True,30,2026-07,Voice infra is product-critical. 65% on single SIP provider.
3,Recruitment & Talent Acquisition,"€6,800K",35%,High,False,45,2026-12,Hiring 170+ roles (380 to 600). Agency fees 20-25% for senior AI talent.
1,AI/ML APIs & Data Services,"€9,200K",55%,High,False,14,2026-06,LLM pricing volatility. Token cost directly impacts COGS.
6,Facilities & Office,"€4,800K",45%,High,False,90,2028-06,"2 new office buildouts (SF, Madrid) in 2026. Lease overlap costs."
5,Marketing & Events,"€5,500K",28%,Medium,False,30,Various,New CMO restructuring spend. US market entry needs 3x budget vs DACH.
8,Hardware & Equipment,"€2,400K",38%,Medium,False,56,N/A,GPU lead times 8-16 weeks. Onboarding kits for 170+ new hires.
4,Professional Services,"€3,200K",40%,Medium,False,21,2026-03,"EU AI Act compliance accelerating. Multi-jurisdiction legal (DE, US, ES)."
7,Travel & Entertainment,"€3,200K",30%,Low,False,3,2026-09,5-office structure driving 80% YoY travel growth.
2,IT Software & SaaS,"€4,200K",22%,Low,False,7,Various,License scaling costs with 40% headcount growth. Shadow IT in new offices.


---
## 7. Next Steps & Ideas

Now that you have the base model, here are things you can try:

### Easy (good Python practice)
- Change the spend numbers in the data model and re-run all cells
- Add a new category (e.g. "Insurance" or "Consulting")
- Change the color palette
- Add a new column like `budget_k` and compute variance (`spend - budget`)

### Intermediate
- Load real data from a CSV file instead of hardcoded numbers:
  ```python
  df = pd.read_csv('spend_export.csv')
  ```
- Add filtering with `ipywidgets` (dropdown to select category/year)
- Calculate rolling averages with `df['spend'].rolling(2).mean()`

### Advanced
- Use **Prophet** or **ARIMA** for time-series spend forecasting
- Connect to **SAP Ariba** or **Coupa** API for live data
- Build a **Streamlit** app to share with stakeholders:
  ```bash
  pip install streamlit
  streamlit run app.py
  ```